Create a data frame containing information about pharmaceutical products that include a given drug. The data frame should include the drug ID, product name, manufacturer, National Drug Code (NDC) from the U.S. registry, dosage form, route of administration, dosage information, country, and the regulatory agency responsible for product registration.

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

file_path = 'drugbank_partial.xml'
namespace = {'drugbank': 'http://www.drugbank.ca'}
data = []
depth = 0

# Parse the XML file and extract relevant information
for event, elem in ET.iterparse(file_path, events=('start', 'end')):
    if event == 'start':
        depth += 1
    elif event == 'end':
        depth -= 1

    # Process each drug element when the end tag is encountered
    if elem.tag == f"{{{namespace['drugbank']}}}drug" and event == 'end' and depth == 1:
        drugbank_id_elem = elem.find('drugbank:drugbank-id[@primary="true"]', namespace)
        if drugbank_id_elem is not None:
            drugbank_id = drugbank_id_elem.text
            products_elem = elem.find('drugbank:products', namespace)
            if products_elem is not None:
                # Extract product information for each product element
                for product in products_elem.findall('drugbank:product', namespace):
                    product_dict = {'drugbank-id': drugbank_id}
                    for field in ['name', 'labeller', 'ndc-product-code', 'dosage-form', 'route', 'strength', 'country', 'source']:
                        field_elem = product.find(f'drugbank:{field}', namespace)
                        if field_elem is not None:
                            product_dict[field] = field_elem.text
                    data.append(product_dict)
        elem.clear()

# Create a DataFrame from the extracted data
df = pd.DataFrame(data)
df

,drugbank-id,name,labeller,ndc-product-code,dosage-form,route,strength,country,source
0,DB00001,Refludan,Bayer,50419-150,Powder,Intravenous,50 mg/1mL,US,FDA NDC
1,DB00001,Refludan,Bayer,None,"Powder, for solution",Intravenous,50 mg / vial,Canada,DPD
2,DB00001,Refludan,Celgene Europe Limited,None,"Injection, solution, concentrate",Intravenous,50 mg,EU,EMA
3,DB00001,Refludan,Celgene Europe Limited,None,"Injection, solution, concentrate",Intravenous,50 mg,EU,EMA
4,DB00001,Refludan,Celgene Europe Limited,None,"Injection, solution, concentrate",Intravenous,20 mg,EU,EMA
...,...,...,...,...,...,...,...,...,...
4579,DB00108,Tysabri,Elan Pharmaceuticals,59075-730,Injection,Intravenous,300 mg/15mL,US,FDA NDC
4580,DB00108,Tysabri,Biogen Inc.,64406-008,Injection,Intravenous,300 mg/15mL,US,FDA NDC
4581,DB00108,Tysabri,Biogen,None,Solution,Intravenous,300 mg / 15 mL,Canada,DPD
4582,DB00108,Tysabri,Biogen Netherlands B.V.,None,"Injection, solution, concentrate",Intravenous,300 mg,EU,EMA
